# Cape Jones — Lighting Gag Reel (Colab T4 GPU)

Renders a first-pass subset of the living-room LED looks. Run cells top to bottom.

**GPU:** In Runtime → Change runtime type, select a T4 GPU if available.

This preview covers eight of the repository's fifteen canonical LED presets; it is not the full 32-grade pass.

In [ ]:
# Cell 1 — Verify GPU availability
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2 — Install Blender 4.0.2
!wget -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!tar -xf blender-4.0.2-linux-x64.tar.xz
!blender-4.0.2-linux-x64/blender --version

In [ ]:
# Cell 3 — Download the living room scene
!wget -q https://raw.githubusercontent.com/mhvnsnt/cape-jones/main/3d-assets/living_room.blend -O living_room.blend
!test -s living_room.blend && ls -lh living_room.blend

In [ ]:
# Cell 4 — Write a guarded render script
%%writefile lighting_reel.py
"""Cape Jones living-room LED preview renderer.
Run from the directory containing living_room.blend:
  blender -b living_room.blend --python lighting_reel.py
"""
import bpy

# Deliberate first-pass subset of the canonical 15 LED presets.
# Each look occupies 72 frames (3 seconds at 24 fps).
SEG = 72
PRESETS = [
    ("purple-pulse", [(0.45, 0.10, 0.90), (0.30, 0.05, 0.70), (0.45, 0.10, 0.90)]),
    ("chill-blue", [(0.10, 0.25, 0.95), (0.05, 0.15, 0.80), (0.10, 0.25, 0.95)]),
    ("ember-glow", [(0.95, 0.15, 0.10), (0.70, 0.08, 0.05), (0.95, 0.15, 0.10)]),
    ("ghost-mode", [(0.10, 0.90, 0.85), (0.05, 0.70, 0.65), (0.10, 0.90, 0.85)]),
    ("ocean-calm", [(0.05, 0.45, 0.85), (0.02, 0.30, 0.65), (0.05, 0.45, 0.85)]),
    ("sunset-cycle", [(0.95, 0.45, 0.10), (0.90, 0.20, 0.55), (0.95, 0.45, 0.10)]),
    ("rainbow-smooth", [(0.90, 0.10, 0.10), (0.10, 0.90, 0.10), (0.10, 0.10, 0.90), (0.90, 0.10, 0.10)]),
    ("blacklight-flicker", [(0.35, 0.05, 0.80), (0.20, 0.02, 0.50), (0.35, 0.05, 0.80)]),
]

def require(condition, message):
    if not condition:
        raise RuntimeError("Lighting reel setup failed: " + message)

# Blender is already running with living_room.blend loaded; do not reopen it.
led_mat = bpy.data.materials.get("LEDStripMat")
require(led_mat is not None, "material 'LEDStripMat' was not found")
require(led_mat.use_nodes and led_mat.node_tree is not None,
        "'LEDStripMat' must have nodes enabled")
led_node = led_mat.node_tree.nodes.get("Principled BSDF")
require(led_node is not None, "Principled BSDF node was not found in 'LEDStripMat'")

emission_socket = led_node.inputs.get("Emission Color")
if emission_socket is None:
    emission_socket = led_node.inputs.get("Emission")
require(emission_socket is not None,
        "Principled BSDF has neither 'Emission Color' nor 'Emission' input")

room_light = bpy.data.objects.get("RoomLight")
require(room_light is not None and room_light.type == "LIGHT",
        "light object 'RoomLight' was not found")

# Use private copies so preview animation does not alter shared source datablocks.
source_mat = led_mat
render_mat = source_mat.copy()
render_mat.name = source_mat.name + "_LightingReelPreview"
for obj in bpy.data.objects:
    for slot in obj.material_slots:
        if slot.material == source_mat:
            slot.material = render_mat
led_node = render_mat.node_tree.nodes.get("Principled BSDF")
emission_socket = led_node.inputs.get("Emission Color")
if emission_socket is None:
    emission_socket = led_node.inputs.get("Emission")
node_tree = render_mat.node_tree
if node_tree.animation_data:
    node_tree.animation_data_clear()

room_light.data = room_light.data.copy()
if room_light.data.animation_data:
    room_light.data.animation_data_clear()

scene = bpy.context.scene
scene.render.engine = "CYCLES"
scene.cycles.device = "GPU"
scene.cycles.samples = 32
scene.cycles.use_denoising = True
scene.cycles.denoiser = "OPTIX"
scene.render.resolution_x = 1280
scene.render.resolution_y = 720
scene.render.resolution_percentage = 100
scene.render.fps = 24
scene.render.image_settings.file_format = "FFMPEG"
scene.render.image_settings.codec = "H264"
scene.render.ffmpeg.format = "MPEG4"
scene.frame_start = 1
scene.frame_end = SEG * len(PRESETS)

for i, (name, colors) in enumerate(PRESETS):
    base = i * SEG + 1
    denominator = max(len(colors) - 1, 1)
    for j, rgb in enumerate(colors):
        # Keep the final keyframe inside this preset's segment; no shared
        # boundary frame with the next preset.
        frame = base + round(j * (SEG - 1) / denominator)
        emission_socket.default_value = (*rgb, 1.0)
        emission_socket.keyframe_insert(data_path="default_value", frame=frame)
        room_light.data.color = rgb
        room_light.data.keyframe_insert(data_path="color", frame=frame)
    print(f"Preset {i + 1}/{len(PRESETS)}: {name} (frames {base}-{base + SEG - 1})")

action = node_tree.animation_data.action if node_tree.animation_data else None
if action:
    for fcurve in action.fcurves:
        for key in fcurve.keyframe_points:
            key.interpolation = "BEZIER"

scene.render.filepath = "/content/cape-jones-lighting-reel.mp4"
print(f"Rendering {len(PRESETS)} presets / {scene.frame_end} frames at 24 fps")
bpy.ops.render.render(animation=True)
print(f"RENDER_COMPLETE: {scene.render.filepath} ({scene.frame_end / scene.render.fps:.1f}s)")


In [ ]:
# Cell 5 — Render and preserve Blender's exit status
!bash -o pipefail -c 'blender-4.0.2-linux-x64/blender -b living_room.blend --python lighting_reel.py 2>&1 | tee render.log | tail -30'

In [ ]:
# Cell 6 — Verify output before download
!test -s /content/cape-jones-lighting-reel.mp4 && ffprobe -v error -show_entries format=duration:stream=width,height,r_frame_rate -of default=noprint_wrappers=1 /content/cape-jones-lighting-reel.mp4
from google.colab import files
files.download('/content/cape-jones-lighting-reel.mp4')